# Предсказание на своей картинке

Модель выдаёт балл от 0 до 1 и карту, где она видит генерацию. Порог берётся из `runs/pilot/report.md`,
его пишет `scripts/run_test.py`: выше порога значит сгенерирована, при 1 % ложных срабатываний на реальных фото.

In [ ]:
import os, re, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from pathlib import Path
import yaml, torch
import matplotlib.pyplot as plt
from src.model import Detector
from src.data import ImageDataset

cfg = yaml.safe_load(open('configs/train.yaml'))
out = Path(cfg['train']['out'])
size = cfg['train']['size']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Detector.load(out / 'best.pt', device)

# порог из отчёта теста (в старых отчётах по разделу на режим: берём native); без отчёта 0.5
text = (out / 'report.md').read_text() if (out / 'report.md').exists() else ''
threshold = float(dict(re.findall(r'## protocol: (\w+)\s+threshold ([\d.]+)', text)).get('native')
                  or (re.findall(r'threshold ([\d.]+)', text) or [0.5])[0])
threshold

Картинка берётся целиком в исходном разрешении и сжимается в JPEG q90, как на обучении. Модель смотрит её кусками 512×512,
как обучающие кадры, с перекрытием наполовину и усредняет карту в перекрытиях, поэтому швов между кусками нет.

In [ ]:
@torch.no_grad()
def predict(path):
    item = ImageDataset([{'path': str(path), 'label': 0}], model.patch_size)[0]
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=device == 'cuda'):
        patch_logits, image_logit = model.full(item['image'][None].to(device), size, step=size // 2)
    prob = torch.sigmoid(image_logit.float()).item()
    return prob, prob > threshold, torch.sigmoid(patch_logits[0, 0].float()).cpu(), item['image']

In [ ]:
path = 'my_images/example.jpg'   # своя картинка

prob, fake, heat, image = predict(path)
print(f"{'сгенерирована' if fake else 'реальная'}: балл {prob:.3f}, порог {threshold:.3f}")
h, w = image.shape[1:]
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(image.permute(1, 2, 0).numpy())
ax[0].set_title(f"{'сгенерирована' if fake else 'реальная'}: {prob:.3f}, порог {threshold:.3f}")
ax[1].imshow(image.permute(1, 2, 0).numpy())
ax[1].imshow(heat.numpy(), cmap='jet', alpha=0.45, vmin=0, vmax=1, extent=(0, w, h, 0), interpolation='bilinear')
ax[1].set_title('где модель видит генерацию')
for a in ax:
    a.axis('off')
plt.show()

In [ ]:
# все картинки из папки
folder = Path('my_images')       # папка, не файл
for p in sorted(folder.iterdir()):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
        prob, fake, _, _ = predict(p)
        print(f"{prob:.3f}  {'FAKE' if fake else 'real'}  {p.name}")